<a href="https://colab.research.google.com/github/Shomyaaraj/MLops-Set5/blob/main/External.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q mlflow fastapi uvicorn nest_asyncio scikit-learn pandas numpy


dataset creation

In [3]:

import numpy as np
import pandas as pd

np.random.seed(42)

NUM_RECORDS = 250
TANK_CAPACITY = 100000

def generate_dataset(version="v1"):

    tank_level = np.random.uniform(20000, 100000, NUM_RECORDS)
    households = np.random.randint(50, 150, NUM_RECORDS)

    multiplier = 1.35 if version == "v2" else 1.0

    consumption = np.random.uniform(15, 30, NUM_RECORDS) * multiplier
    pumping_duration = np.random.uniform(0, 4, NUM_RECORDS)
    pumping_rate = np.random.uniform(8000, 12000, NUM_RECORDS)
    leakage_rate = np.random.uniform(100, 500, NUM_RECORDS)
    day_type = np.random.randint(0, 2, NUM_RECORDS)

    total_demand = households * consumption * 10
    total_pumping = pumping_duration * pumping_rate
    total_leakage = leakage_rate * 10

    morning_level = tank_level - total_demand + total_pumping - total_leakage
    morning_level = np.minimum(morning_level, TANK_CAPACITY)

    shortage = (morning_level < 15000).astype(int)

    df = pd.DataFrame({
        "tank_level_8pm": tank_level,
        "households": households,
        "avg_hourly_consumption": consumption,
        "pumping_duration": pumping_duration,
        "pumping_rate": pumping_rate,
        "leakage_rate": leakage_rate,
        "day_type": day_type,
        "morning_water_shortage": shortage
    })

    return df

df_v1 = generate_dataset("v1")
df_v2 = generate_dataset("v2")

df_v1.to_csv("water_tank_v1.csv", index=False)
df_v2.to_csv("water_tank_v2.csv", index=False)

print("Dataset V1 Created:", df_v1.shape)
print("Dataset V2 Created:", df_v2.shape)

print("V1 Shortage Rate:", df_v1["morning_water_shortage"].mean())
print("V2 Shortage Rate:", df_v2["morning_water_shortage"].mean())

display(df_v2.head())


Dataset V1 Created: (250, 8)
Dataset V2 Created: (250, 8)
V1 Shortage Rate: 0.092
V2 Shortage Rate: 0.124


,tank_level_8pm,households,avg_hourly_consumption,pumping_duration,pumping_rate,leakage_rate,day_type,morning_water_shortage
0,76723.700938,132,23.762972,0.509816,8602.085967,172.695676,0,0
1,28550.153848,118,33.160454,2.830753,11267.027720,274.203050,1,0
2,65384.977570,81,30.297369,3.891105,8829.236013,232.839746,0,0
3,40525.022696,93,23.859612,3.304156,11129.553318,236.721482,0,0
4,97034.150019,114,34.139703,2.797262,10938.311692,395.610623,1,0


data preprocessing and feature engineering

In [4]:

def preprocess_data(df):

    df = df.copy()

    df["expected_overnight_demand"] = (
        df["households"] *
        df["avg_hourly_consumption"] * 10
    )

    df["expected_pump_supply"] = (
        df["pumping_duration"] *
        df["pumping_rate"]
    )

    df["projected_water_buffer"] = (
        df["tank_level_8pm"]
        - df["expected_overnight_demand"]
        - df["leakage_rate"] * 10
        + df["expected_pump_supply"]
    )

    return df

engineered_data = preprocess_data(df_v2)

engineered_data.to_csv("feature_store.csv", index=False)

print("Feature Engineering Completed!")
display(engineered_data.head())


Feature Engineering Completed!


,tank_level_8pm,households,avg_hourly_consumption,pumping_duration,pumping_rate,leakage_rate,day_type,morning_water_shortage,expected_overnight_demand,expected_pump_supply,projected_water_buffer
0,76723.700938,132,23.762972,0.509816,8602.085967,172.695676,0,0,31367.122385,4385.479663,48015.101453
1,28550.153848,118,33.160454,2.830753,11267.027720,274.203050,1,0,39129.335255,31894.175961,18572.964052
2,65384.977570,81,30.297369,3.891105,8829.236013,232.839746,0,0,24540.869115,34355.486970,72871.197963
3,40525.022696,93,23.859612,3.304156,11129.553318,236.721482,0,0,22189.439583,36773.782673,52742.150964
4,97034.150019,114,34.139703,2.797262,10938.311692,395.610623,1,0,38919.261692,30597.318178,84756.100274


model training

In [5]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix
)

features = [
    "tank_level_8pm",
    "households",
    "avg_hourly_consumption",
    "pumping_duration",
    "pumping_rate",
    "leakage_rate",
    "day_type",
    "expected_overnight_demand",
    "expected_pump_supply",
    "projected_water_buffer"
]

X = engineered_data[features]
y = engineered_data["morning_water_shortage"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42)
}

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results[name] = {
        "model": model,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    }

    print("\n", name)
    print("Accuracy:", results[name]["Accuracy"])
    print("Precision:", results[name]["Precision"])
    print("Recall:", results[name]["Recall"])
    print("F1 Score:", results[name]["F1 Score"])
    print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))

best_name = max(results, key=lambda x: results[x]["F1 Score"])
best_model = results[best_name]["model"]

print("\nBest Model Selected:", best_name)



 Decision Tree
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[44  0]
 [ 0  6]]

 Random Forest
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
 [[44  0]
 [ 0  6]]

Best Model Selected: Decision Tree


tracking

In [7]:

import mlflow
import mlflow.sklearn

mlflow.set_experiment("Water_Shortage_Prediction")

with mlflow.start_run():

    mlflow.log_param("model_name", best_name)
    mlflow.log_param("dataset_version", "v2")

    mlflow.log_metric("accuracy", results[best_name]["Accuracy"])
    mlflow.log_metric("precision", results[best_name]["Precision"])
    mlflow.log_metric("recall", results[best_name]["Recall"])
    mlflow.log_metric("f1_score", results[best_name]["F1 Score"])

    # Trust the Scikit-learn tree structure
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="water_shortage_model",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

print("Model successfully saved in MLflow!")
print("Experiment: Water_Shortage_Prediction")
print("Selected Model:", best_name)


Model successfully saved in MLflow!
Experiment: Water_Shortage_Prediction
Selected Model: Decision Tree


CICD testing

In [8]:

def run_tests():

    assert df_v2["tank_level_8pm"].between(0, TANK_CAPACITY).all()
    assert (df_v2["households"] > 0).all()
    assert (df_v2["leakage_rate"] >= 0).all()

    assert len(X_test) > 0

    predictions = best_model.predict(X_test)

    assert len(predictions) == len(X_test)

    print("All CI/CD Tests Passed Successfully!")

run_tests()


All CI/CD Tests Passed Successfully!


fastAPI model depmentloy

In [9]:

from fastapi import FastAPI
from pydantic import BaseModel, Field
import nest_asyncio
import uvicorn
import threading
import time
import requests

app = FastAPI(title="Water Shortage Prediction API")

class TankData(BaseModel):
    tank_level_8pm: float = Field(ge=0, le=100000)
    households: int = Field(gt=0)
    avg_hourly_consumption: float = Field(ge=0)
    pumping_duration: float = Field(ge=0, le=10)
    pumping_rate: float = Field(ge=0)
    leakage_rate: float = Field(ge=0)
    day_type: int = Field(ge=0, le=1)

@app.get("/")
def home():
    return {"message": "Water Shortage Prediction API is running"}

@app.post("/predict")
def predict(data: TankData):

    demand = data.households * data.avg_hourly_consumption * 10
    pumping = data.pumping_duration * data.pumping_rate

    buffer = (
        data.tank_level_8pm
        - demand
        - data.leakage_rate * 10
        + pumping
    )

    input_data = pd.DataFrame([{
        "tank_level_8pm": data.tank_level_8pm,
        "households": data.households,
        "avg_hourly_consumption": data.avg_hourly_consumption,
        "pumping_duration": data.pumping_duration,
        "pumping_rate": data.pumping_rate,
        "leakage_rate": data.leakage_rate,
        "day_type": data.day_type,
        "expected_overnight_demand": demand,
        "expected_pump_supply": pumping,
        "projected_water_buffer": buffer
    }])[features]

    prediction = best_model.predict(input_data)[0]
    probability = best_model.predict_proba(input_data)[0][1]

    return {
        "Prediction": "Shortage Risk" if prediction == 1 else "Adequate Reserve",
        "Shortage_Probability": round(float(probability), 4),
        "Projected_Water_Buffer": round(float(buffer), 2)
    }

nest_asyncio.apply()

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

time.sleep(3)

print("FastAPI Server Started!")


INFO:     Started server process [781]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI Server Started!


test the API

In [10]:


sample_input = {
    "tank_level_8pm": 40000,
    "households": 120,
    "avg_hourly_consumption": 25,
    "pumping_duration": 1.5,
    "pumping_rate": 9500,
    "leakage_rate": 250,
    "day_type": 1
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input
)

print("API Status Code:", response.status_code)
print("Prediction Result:")
print(response.json())


INFO:     127.0.0.1:60780 - "POST /predict HTTP/1.1" 200 OK
API Status Code: 200
Prediction Result:
{'Prediction': 'Adequate Reserve', 'Shortage_Probability': 0.0, 'Projected_Water_Buffer': 21750.0}
